# Skin Problem Detection — Draft
Notebook นี้เรียงลำดับตามแนวทางของ notebook อาจารย์ แต่ใช้ path แบบ relative และไม่แก้ข้อมูลใน `datasets/raw/`

Draft วันที่ 30 กันยายนเป็น Object Detection จำนวน 4 classes: `acne`, `wrinkle`, `dark_spot`, `enlarged_pore` ส่วน `normal_skin` ยังไม่รวมเพราะไม่มี bounding box จริง

## 1. Project Setup

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# รองรับการเปิด notebook จาก root หรือจากโฟลเดอร์ notebooks
candidate = Path.cwd().resolve()
PROJECT_ROOT = candidate if (candidate / 'scripts').is_dir() else candidate.parent
print('Project root:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])

## 2. Dataset Information
อ่าน mapping ที่อนุมัติและยืนยันว่า raw data จะไม่ถูกแก้ไข

In [ ]:
import yaml

classes_config = yaml.safe_load((PROJECT_ROOT / 'configs/classes.yaml').read_text(encoding='utf-8'))
classes_config

## 3. Dataset Preparation
Cell นี้สร้าง derived dataset ใหม่จาก COCO source โดยไม่แก้ raw data

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts/prepare_detection_dataset.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
report_path = PROJECT_ROOT / 'datasets/processed/skin_detection/preparation_report.json'
preparation_report = json.loads(report_path.read_text(encoding='utf-8'))
preparation_report['splits']

## 4. Dataset Validation
Validator ตรวจ pairing, class IDs, coordinates, รูปเสีย, negative images และ duplicate leakage

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts/validate_dataset.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

## 5. Visualize Bounding Boxes
สุ่มแบบ deterministic สูงสุด class ละ 5 ภาพ เพื่อให้คนตรวจกรอบก่อน train

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts/visualize_annotations.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
from IPython.display import Image as IPyImage, display

sample_paths = sorted((PROJECT_ROOT / 'results/annotation_samples').glob('*/*.jpg'))
for sample_path in sample_paths:
    print(sample_path.relative_to(PROJECT_ROOT))
    display(IPyImage(filename=str(sample_path), width=420))

## 6. Model 1 — YOLO11n (Phase 3 Placeholder)
จะโหลด pretrained `yolo11n.pt` และ train ด้วย `configs/yolo11n.yaml` หลังผู้ใช้อนุมัติ Phase 3 เท่านั้น **ห้ามรัน training ใน Phase 2**

## 7. Model 2 — YOLO11s (Phase 3 Placeholder)
จะใช้ dataset และ initial settings เดียวกับ YOLO11n เพื่อให้เปรียบเทียบได้อย่างยุติธรรม **ยังไม่ดาวน์โหลด weights และยังไม่ train**

## 8. Evaluation (Phase 3 Placeholder)
หลัง train จะโหลด `best.pt` ของแต่ละโมเดลและรายงาน Precision, Recall, F1, mAP@0.5, mAP@0.5:0.95, confusion matrix, F1 curve, `results.png` และ validation predictions

## 9. Model Comparison (Phase 3 Placeholder)
เปรียบเทียบ YOLO11n กับ YOLO11s บน test set และ hardware เดียวกัน โดยใช้ mAP, Precision, Recall, F1, training time, inference time, FPS, parameters และ model size

## 10. Inference (Phase 3 Placeholder)
จะแสดง Bounding Box + Class Name + Confidence บนภาพที่ไม่อยู่ใน train/val/test หลังผ่านการประเมินโมเดลแล้ว